# 04 ・ 讓 AI 根據你的資料回答

## 這一章要做什麼

把上一章整理好的電影清單交給 Gemini，讓它回答「推薦一部高分電影」
這類問題 —— 而且是根據**今天實際在上映的片**，不是憑記憶。

## 核心觀念

語言模型不知道今天有哪些電影在上映。它的知識停在訓練資料的時間點，
你問它「今天有什麼好看的」，它只能猜。

解法很簡單：**把資料放進 prompt 裡**。

```
你是電影推薦助手。目前時間：2026年8月28日。
以下是目前上映的電影：
1. 藍色監獄（評分 7.2，動畫、動作，上映影城: 秀泰、美麗華）
2. 蜘蛛人：重生日（評分 6.9，動作、科幻…）
...
請根據這些資料回答使用者的問題。
```

這就是 **RAG（檢索增強生成）** 最基本的形式：
先把相關資料找出來，塞進 prompt，再讓模型基於這些資料回答。
真正的 RAG 系統會用向量檢索挑出相關片段，原理是一樣的。

## 本章產出

`movieapp/gemini.py`：

```python
gemini.build_system_prompt(movies, genre_names)  # 組裝背景資料
gemini.ask(message, system=…, session_id=…)      # 問一句話
```

---
## 0 ・ 開場與備料

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")
from movieapp.config import setup; setup(requires=["sources", "tmdb", "merge"])

In [ ]:
from movieapp import merge, sources, tmdb

by_source, _ = sources.titles_by_source()
genre_names, _ = tmdb.genres()
enriched = {k: tmdb.enrich(v, source=k, workers=8)[0] for k, v in by_source.items()}
movies = merge.merge_sources(enriched)
print(f"手上有 {len(movies)} 部電影，準備交給 AI")

---
## 1 ・ 組裝 system prompt

把電影清單整理成一段文字。幾個設計重點：

- **附上目前時間**，模型才能回答「今天上映」這類問題
- **附上上映影城**，才能回答「秀泰有什麼好看的」
- **簡介要截短**（取前 200 字），不然 prompt 會太長
- **明講「資料裡沒有的不要編造」**，減少模型自己掰
- **限制電影數量**，50 部以上 prompt 就相當長了

In [ ]:
%%writefile ../movieapp/gemini.py
"""Gemini 聊天：把目前的電影清單交給模型，讓它根據這些資料回答。

用的是 Interactions API。它和一般 chat API 最大的差別是
「對話記憶由伺服器保管」：每次回應會給一個 interaction id，
下一次帶上 previous_interaction_id 就能接續前文，
不必自己把整段對話重新送一遍。

本檔案由 notebooks/04_Gemini對話.ipynb 的 %%writefile 產生。
要修改請回去改那一格，不要直接編輯這裡。
"""

import time
from datetime import datetime

from movieapp import config
from movieapp.http import fetch_json

API_URL = "https://generativelanguage.googleapis.com/v1beta/interactions"

# 模型會下架。gemini-2.5-flash 和 gemini-2.0-flash 都已經對新使用者關閉，
# 呼叫會收到 404 not_found。主模型不可用時自動往後試。
MODELS = ["gemini-3.6-flash", "gemini-flash-latest"]

# 每個 session_id 目前接到哪一輪對話
_INTERACTION_IDS = {}

# 一次對話最多帶幾部電影進 system prompt，避免 prompt 過長
MAX_MOVIES_IN_PROMPT = 60

# 模型回一段話要花的時間比一般 API 長很多
REPLY_TIMEOUT = 90


def reset_session(session_id=None):
    """清掉對話記憶，讓下一句重新開始。"""
    if session_id is None:
        _INTERACTION_IDS.clear()
    else:
        _INTERACTION_IDS.pop(session_id, None)


def build_system_prompt(movies, genre_names=None, now=None):
    """把目前的電影整理成一段文字，當作模型的背景知識。

    這就是最陽春的 RAG：模型本身不知道今天有哪些電影在上映，
    我們把資料放進 system prompt，它才能根據實際片單回答。
    """
    genre_names = genre_names or {}
    if not movies:
        return ""

    lines = []
    for index, movie in enumerate(movies[:MAX_MOVIES_IN_PROMPT], 1):
        meta = movie.get("meta") or {}
        parts = []
        if meta.get("title"):
            parts.append(f"片名: {meta['title']}")
        if meta.get("release_date"):
            parts.append(f"上映日期: {meta['release_date']}")
        if meta.get("vote_average") is not None:
            parts.append(f"評分: {meta['vote_average']:.1f}")
        if meta.get("genre_ids"):
            names = "、".join(genre_names.get(g, str(g)) for g in meta["genre_ids"])
            parts.append(f"類型: {names}")
        if movie.get("sources"):
            from movieapp.merge import source_labels

            parts.append(f"上映影城: {'、'.join(source_labels(movie))}")
        if meta.get("overview"):
            parts.append(f"簡介: {meta['overview'][:200]}")
        detail = f"（{'，'.join(parts)}）" if parts else ""
        lines.append(f"{index}. {movie.get('title')}{detail}")

    stamp = (now or datetime.now()).strftime("%Y年%m月%d日 %H:%M:%S")
    return (
        f"你是電影推薦助手，請用繁體中文回答。目前時間：{stamp}。\n"
        "以下是目前頁面顯示的電影資料，請依據這些資料回答使用者的問題"
        "（例如推薦、比較、說明）。資料裡沒有的電影不要編造。\n"
        + "\n".join(lines)
    )


def _describe_error(error):
    """把 API 錯誤翻譯成看得懂的說明。"""
    text = str(error or "")
    if "429" in text:
        return (
            "Gemini 配額已用完（429）。免費額度是綁在金鑰上計算的，"
            "全班共用一把金鑰時很容易同時撞到。可以等幾分鐘再試，或改用自己的金鑰。"
        )
    if "404" in text or "not found" in text.lower():
        return f"模型不存在或已下架：{text}"
    return text


def extract_reply(data):
    """從 Interactions API 的回應裡取出模型講的話。

    回應是一連串 steps，我們只要 type 為 model_output 的文字部分。
    """
    steps = (data or {}).get("steps") or []
    return "".join(
        part.get("text", "")
        for step in steps
        if step.get("type") == "model_output"
        for part in (step.get("content") or [])
        if part.get("type") == "text"
    ).strip()


def ask(message, system=None, session_id=None, models=None, retries=1):
    """問一句話，回傳 (reply, error)。

    session_id 相同時會自動接續前一輪對話。
    """
    message = str(message or "").strip()
    if not message:
        return "", "沒有訊息內容"

    payload_base = {"input": message}
    if system:
        payload_base["system_instruction"] = system
    if session_id and session_id in _INTERACTION_IDS:
        payload_base["previous_interaction_id"] = _INTERACTION_IDS[session_id]

    last_error = "Gemini 無回應"
    for model in models or MODELS:
        for attempt in range(retries + 1):
            payload = dict(payload_base, model=model)
            data, error = fetch_json(
                API_URL,
                method="POST",
                headers={
                    "x-goog-api-key": config.gemini_key(),
                    "Content-Type": "application/json",
                },
                json_body=payload,
                # 語言模型要花時間想，20 秒的預設值不夠
                timeout=REPLY_TIMEOUT,
            )

            if error:
                last_error = error
                # 伺服器忙碌（500）值得重試；配額或模型問題重試沒用
                if "500" in str(error) and attempt < retries:
                    time.sleep(2)
                    continue
                break

            reply = extract_reply(data)
            interaction_id = (data or {}).get("id") or ""
            if session_id and interaction_id:
                _INTERACTION_IDS[session_id] = interaction_id
            if reply:
                return reply, None
            last_error = "Gemini 回應中沒有文字內容"
            break

        # 模型不可用才換下一個試
        if "404" not in str(last_error) and "429" not in str(last_error):
            break

    return "", _describe_error(last_error)


# 畫面上聊天面板的快捷問題
QUICK_PROMPTS = [
    "推薦一部高分電影",
    "推薦黑暗風格的電影",
    "推薦喜劇片",
    "哪部電影評分最高？",
    "推薦今天上映的電影",
    "推薦動作片",
]

In [ ]:
from movieapp import gemini

system = gemini.build_system_prompt(movies, genre_names)
print(f"system prompt 長度：{len(system)} 字\n")
print(system[:700])
print("\n   ...（後面還有）")

---
## 2 ・ 問第一個問題

`ask()` 回傳 `(reply, error)`，和前面幾章一樣的形式。

In [ ]:
reply, error = gemini.ask("哪部電影評分最高？", system=system)
print(reply if not error else f"失敗：{error}")

模型講出來的是**你資料裡的電影**，不是它記憶中的老片。
把 `system` 拿掉再問一次，差別就很明顯：

In [ ]:
reply, error = gemini.ask("哪部電影評分最高？")     # 不給資料
print(reply[:300] if not error else f"失敗：{error}")

沒有資料時它只能給泛泛的答案，或是講一部它記憶中的電影 ——
可能根本沒在上映。這就是為什麼要把資料塞進 prompt。

---
## 3 ・ 多輪對話

Interactions API 的對話記憶由**伺服器**保管。
每次回應會給一個 `id`，下一次帶上 `previous_interaction_id` 就接得起來。

好處是不用每次把整段對話重送一遍（省 token、省頻寬）。
`ask()` 用 `session_id` 幫你記住這件事：

> **多輪對話沒有辦法離線重現。** 第二句的答案取決於伺服器記住的第一句，
> 那是存在 Google 那邊的狀態，不是我們存得下來的東西 ——
> 這也正是 `previous_interaction_id` 這個設計的意義。
> 配額用完時下面會顯示錯誤訊息，那本身也說明了這個機制的特性。

In [ ]:
gemini.reset_session("demo")

reply, error = gemini.ask("推薦一部動畫電影", system=system, session_id="demo")
print("Q1: 推薦一部動畫電影")
print("A1:", (reply or error)[:200], "\n")

# 第二句沒有提電影名字，模型要靠對話記憶才知道「它」是誰
reply, error = gemini.ask("它適合小朋友看嗎？", session_id="demo")
print("Q2: 它適合小朋友看嗎？")
print("A2:", (reply or error)[:200])

In [ ]:
# 記憶就存在這裡，每個 session 一個 interaction id
print(gemini._INTERACTION_IDS)

> **這個記憶存在記憶體裡。** 服務重開就沒了，多開幾個 worker 也不會共享。
> 教學夠用，真的要上線就得換成 Redis 之類的外部儲存。

---
## 4 ・ 配額：這一章最容易卡住的地方

Gemini 免費層的額度是**按金鑰計算**，不是按人或按 IP。
全班共用一把金鑰時，等於三十個人擠同一個額度 ——
大家同時發問就會有人收到 **429**。

`ask()` 會把 429 翻譯成看得懂的說明，而不是丟一串英文給學員：

In [ ]:
print(gemini._describe_error("Gemini 回應錯誤: HTTP 429 quota exceeded"))

撞到的時候有兩個選擇：

1. 等幾分鐘再試 —— 免費層是按時間窗計算的，會自己恢復
2. 用自己的金鑰（<https://aistudio.google.com/apikey> 免費申請），
   把它填進 `.env`，或把 `.env` 裡的值清空讓程式當場問你

重點是**程式不能因為配額用完就崩潰**。看看 `ask()` 怎麼處理：
它把 429 翻譯成一句看得懂的中文，照樣回傳 `(reply, error)`，
呼叫端只要把 error 顯示出來就好。

In [ ]:
# 配額錯誤長什麼樣：故意用一把無效金鑰，看它回什麼
import os

from movieapp import config

real_key = os.environ.get(config.GEMINI_KEY_NAME, "")
os.environ[config.GEMINI_KEY_NAME] = "invalid-key-for-demo"

reply, error = gemini.ask("推薦一部電影")
print("reply:", repr(reply))
print("error:", error)

os.environ[config.GEMINI_KEY_NAME] = real_key
print("\n金鑰已還原：", bool(os.environ.get(config.GEMINI_KEY_NAME)))

**沒有丟例外，程式繼續跑。** 這就是 `(data, error)` 這種回傳形式的價值 ——
第 5 章把它接成網頁服務時，AI 掛掉只會讓聊天框顯示一行紅字，
不會讓整個網站噴 500。

---
## 5 ・ 模型會下架

寫這份教材時發生了一件事：原本設定的 `gemini-2.5-flash`
突然回 404，訊息說「已不再對新使用者開放」。`gemini-2.0-flash` 也一樣。

這不是特例，是雲端 AI 服務的常態。所以 `MODELS` 是一個清單，
主模型不可用時自動往後試：

```python
MODELS = ["gemini-3.6-flash", "gemini-flash-latest"]
```

**寫程式呼叫外部 AI 服務時，模型名稱要當成會變的設定，不是常數。**

---
## 小結

- 模型不知道你的資料，**把資料放進 prompt** 才能根據事實回答 —— 這就是 RAG 的核心
- system prompt 要附上時間、來源，並明講「不要編造」
- Interactions API 的對話記憶由伺服器保管，用 `previous_interaction_id` 接續
- 免費額度**綁金鑰**，多人共用會一起撞 429，錯誤訊息要寫得讓人看得懂
- 模型名稱會下架，要準備 fallback

### 產出

`movieapp/gemini.py`

### 下一章

**05_接成服務** —— 四個模組都做好了。
最後一章把它們接成真的 API 服務，並且在 notebook 裡直接看到完整網頁。